# 01 · Bronze — Ingesta raw del dataset NIES

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Ingestar el fichero Excel del NIES (`2_water quality_eng_v20250925.xlsx`) y almacenarlo
en formato **Delta Lake** como capa Bronze, con metadatos de procedencia añadidos.

## Estructura del notebook
1. Configuración del entorno
2. Lectura y normalización del Excel NIES
3. Conversión a Spark y enriquecimiento con metadatos
4. Escritura en Delta Lake (capa Bronze)
5. Validación de la ingesta

_Revisado: header row confirmado = fila 2 (index 2)._

## 1. Configuración del entorno

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Celda 1 — Importaciones y variables de configuración
# ──────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, IntegerType
)
from datetime import datetime

# ── Parámetros de configuración ──────────────────────────────────────────────
NIES_EXCEL_PATH = "/FileStore/tables/2_water_quality_eng_v20250925.xlsx"
# En Databricks Community Edition el fichero debe subirse con:
#   Data → Add Data → Upload File → luego usar la ruta /FileStore/tables/

CATALOG = "training"            # Catálogo Unity Catalog (detección automática)
SCHEMA  = "kasumigaura"         # Esquema (base de datos)
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"

EXCEL_HEADER_ROW = 2            # El fichero NIES tiene 2 filas de encabezado
INGESTION_TIMESTAMP = datetime.utcnow().isoformat()
DATASET_VERSION = "v20250925"   # Versión del fichero NIES
SOURCE_URL = (
    "https://db.cger.nies.go.jp/gem/moni-e/inter/GEMS/database/kasumi/"
    "contents/datalist.html"
)

print(f"Tabla destino : {TABLE_BRONZE}")
print(f"Timestamp     : {INGESTION_TIMESTAMP}")
print(f"Versión NIES  : {DATASET_VERSION}")

In [ ]:
# TODO: add schema validation and write to Delta Lake